# Data Standardization Before Exporting To MySQL

## Import Libraries And Data

In [1]:
import pandas as pd
import numpy as np


import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
import os
from pathlib import Path

In [2]:
env_path = Path.cwd()/"github_repo_cloned"/"ipo_reality_check"/".env"
env_path

load_dotenv(env_path)

True

In [3]:
data_pref = os.getenv("DATA_PATH_PREF")

In [4]:
vix = pd.read_csv(os.path.join(data_pref,"raw","indiavix.csv"))
nifty = pd.read_csv(os.path.join(data_pref, "raw", "nifty50.csv"))
ipo = pd.read_csv(os.path.join(data_pref, "raw", "ipo_final_clean.csv"))
prices = pd.read_csv(os.path.join(data_pref, "raw", "prices_final_clean.csv"))

In [5]:
vix.head()

,Price,Close,High,Low,Open
0,01-01-2019,15.330000,16.400000,15.15,16.000000
1,02-01-2019,16.389999,16.950001,14.84,15.330000
2,03-01-2019,16.790001,17.150000,15.44,16.389999
3,04-01-2019,16.160000,16.969999,15.49,16.790001
4,07-01-2019,16.299999,16.450001,15.30,16.160000


In [6]:
nifty.head()

,Price,Close,High,Low,Open,Volume
0,02-01-2019,10792.50000,10895.34961,10735.04980,10868.84961,309700
1,03-01-2019,10672.25000,10814.04980,10661.25000,10796.79980,286200
2,04-01-2019,10727.34961,10741.04980,10628.65039,10699.70020,296600
3,07-01-2019,10771.79980,10835.95020,10750.15039,10804.84961,269400
4,08-01-2019,10802.15039,10818.45020,10733.25000,10786.25000,277700


In [7]:
ipo.head()

,company_name,ipo_issue_type,ipo_issue_price_final,listing_date,il_bse_script_code,il_nse_script_symbol,ipo_issue_size_crore,change_in_percentage_listing_day,ildt_open_price,ildt_close_price,ildt_high_price,ildt_low_price,qib,nii,rii,emp,total,last_modified_date,year
0,Prince Pipes & Fittings Ltd.,IPO,178,30-12-2019,542907,PRINCEPIPE,500.00,-6.24,160.00,166.90,177.9,152.50,2.02,0.52,1.63,NaN,1.66,14-07-2026,2019
1,Ujjivan Small Finance Bank Ltd.,IPO,37,12-12-2019,542904,UJJIVANSFB,745.95,51.22,58.75,55.95,62.8,53.05,46.30,473.33,45.29,NaN,99.61,14-07-2026,2019
2,CSB Bank Ltd.,IPO,195,04-12-2019,542867,CSBBANK,409.68,53.87,275.00,300.05,306.8,275.00,25.52,164.27,40.98,NaN,47.88,14-07-2026,2019
3,Vishwaraj Sugar Industries Ltd.,IPO,60,15-10-2019,542852,VISHWARAJ,60.00,0.58,61.20,60.35,61.5,60.25,1.00,1.75,0.59,NaN,1.09,14-07-2026,2019
4,Indian Railway Catering & Tourism Corp.Ltd.,IPO,320,14-10-2019,542830,IRCTC,637.97,127.42,626.00,727.75,743.8,625.00,105.55,351.03,13.28,0.88,109.21,14-07-2026,2019


In [8]:
prices.head()

,company_name,ticker,date,open,high,low,close,volume
0,ACME Solar Holdings Ltd.,ACMESOLAR.NS,13-11-2024,251.000000,276.100006,248.500000,253.149994,22939368
1,ACME Solar Holdings Ltd.,ACMESOLAR.NS,14-11-2024,252.000000,259.899994,227.850006,228.850006,5351111
2,ACME Solar Holdings Ltd.,ACMESOLAR.NS,18-11-2024,236.000000,251.699997,228.250000,251.699997,3943111
3,ACME Solar Holdings Ltd.,ACMESOLAR.NS,19-11-2024,261.049988,276.850006,256.049988,264.149994,9862379
4,ACME Solar Holdings Ltd.,ACMESOLAR.NS,21-11-2024,265.450012,267.049988,237.750000,239.000000,4049771


## Cleaning Data

### Generalized

In [9]:
print(f"Extra In IPO: ")
set(ipo['company_name'])-set(prices['company_name'])
print("-"*50)
print(f"Extra in Prices: ")
set(prices['company_name']) - set(ipo['company_name'])

Extra In IPO: 
--------------------------------------------------
Extra in Prices: 


set()

### Prices

In [10]:
prices.head()

,company_name,ticker,date,open,high,low,close,volume
0,ACME Solar Holdings Ltd.,ACMESOLAR.NS,13-11-2024,251.000000,276.100006,248.500000,253.149994,22939368
1,ACME Solar Holdings Ltd.,ACMESOLAR.NS,14-11-2024,252.000000,259.899994,227.850006,228.850006,5351111
2,ACME Solar Holdings Ltd.,ACMESOLAR.NS,18-11-2024,236.000000,251.699997,228.250000,251.699997,3943111
3,ACME Solar Holdings Ltd.,ACMESOLAR.NS,19-11-2024,261.049988,276.850006,256.049988,264.149994,9862379
4,ACME Solar Holdings Ltd.,ACMESOLAR.NS,21-11-2024,265.450012,267.049988,237.750000,239.000000,4049771


In [11]:
print(f"Rows in Prices:{prices.shape[0]}\
       \nColumns in Prices: {prices.shape[1]}\
         \nUnique Company Names in Prices: {prices['company_name'].nunique()}")

Rows in Prices:251004       
Columns in Prices: 8         
Unique Company Names in Prices: 302


In [12]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251004 entries, 0 to 251003
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   company_name  251004 non-null  object 
 1   ticker        251004 non-null  object 
 2   date          251004 non-null  object 
 3   open          251004 non-null  float64
 4   high          251004 non-null  float64
 5   low           251004 non-null  float64
 6   close         251004 non-null  float64
 7   volume        251004 non-null  int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 15.3+ MB


In [13]:
prices['date'] = pd.to_datetime(prices['date'], format='%d-%m-%Y').dt.date

In [14]:
prices[['open','high','low','close']] = prices[['open','high','low','close']].round(2)

In [15]:
prices.drop(columns='ticker',inplace=True)

In [16]:
prices.isnull().sum().reset_index(name='null_values')

,index,null_values
0,company_name,0
1,date,0
2,open,0
3,high,0
4,low,0
5,close,0
6,volume,0


In [17]:
prices.rename(columns={'company_name':'company'},inplace=True)

In [18]:
prices.head()

,company,date,open,high,low,close,volume
0,ACME Solar Holdings Ltd.,2024-11-13,251.00,276.10,248.50,253.15,22939368
1,ACME Solar Holdings Ltd.,2024-11-14,252.00,259.90,227.85,228.85,5351111
2,ACME Solar Holdings Ltd.,2024-11-18,236.00,251.70,228.25,251.70,3943111
3,ACME Solar Holdings Ltd.,2024-11-19,261.05,276.85,256.05,264.15,9862379
4,ACME Solar Holdings Ltd.,2024-11-21,265.45,267.05,237.75,239.00,4049771


In [19]:
prices[prices[['company','date']].duplicated()]

,company,date,open,high,low,close,volume


In [20]:
prices.to_csv(os.path.join(data_pref,"clean","prices.csv"),index=False)

### IPO

In [21]:
ipo.head()

,company_name,ipo_issue_type,ipo_issue_price_final,listing_date,il_bse_script_code,il_nse_script_symbol,ipo_issue_size_crore,change_in_percentage_listing_day,ildt_open_price,ildt_close_price,ildt_high_price,ildt_low_price,qib,nii,rii,emp,total,last_modified_date,year
0,Prince Pipes & Fittings Ltd.,IPO,178,30-12-2019,542907,PRINCEPIPE,500.00,-6.24,160.00,166.90,177.9,152.50,2.02,0.52,1.63,NaN,1.66,14-07-2026,2019
1,Ujjivan Small Finance Bank Ltd.,IPO,37,12-12-2019,542904,UJJIVANSFB,745.95,51.22,58.75,55.95,62.8,53.05,46.30,473.33,45.29,NaN,99.61,14-07-2026,2019
2,CSB Bank Ltd.,IPO,195,04-12-2019,542867,CSBBANK,409.68,53.87,275.00,300.05,306.8,275.00,25.52,164.27,40.98,NaN,47.88,14-07-2026,2019
3,Vishwaraj Sugar Industries Ltd.,IPO,60,15-10-2019,542852,VISHWARAJ,60.00,0.58,61.20,60.35,61.5,60.25,1.00,1.75,0.59,NaN,1.09,14-07-2026,2019
4,Indian Railway Catering & Tourism Corp.Ltd.,IPO,320,14-10-2019,542830,IRCTC,637.97,127.42,626.00,727.75,743.8,625.00,105.55,351.03,13.28,0.88,109.21,14-07-2026,2019


In [22]:
print(f"Rows in IPO:{ipo.shape[0]}\
       \nColumns in IPO: {ipo.shape[1]}\
         \nUnique Company Names in IPO: {ipo['company_name'].nunique()}")

Rows in IPO:302       
Columns in IPO: 19         
Unique Company Names in IPO: 302


In [23]:
ipo['ipo_issue_type'].nunique()

1

In [24]:
ipo.drop(columns = ['ipo_issue_type','last_modified_date','year','il_nse_script_symbol','il_bse_script_code'],inplace=True)

In [25]:
ipo.head()

,company_name,ipo_issue_price_final,listing_date,ipo_issue_size_crore,change_in_percentage_listing_day,ildt_open_price,ildt_close_price,ildt_high_price,ildt_low_price,qib,nii,rii,emp,total
0,Prince Pipes & Fittings Ltd.,178,30-12-2019,500.00,-6.24,160.00,166.90,177.9,152.50,2.02,0.52,1.63,NaN,1.66
1,Ujjivan Small Finance Bank Ltd.,37,12-12-2019,745.95,51.22,58.75,55.95,62.8,53.05,46.30,473.33,45.29,NaN,99.61
2,CSB Bank Ltd.,195,04-12-2019,409.68,53.87,275.00,300.05,306.8,275.00,25.52,164.27,40.98,NaN,47.88
3,Vishwaraj Sugar Industries Ltd.,60,15-10-2019,60.00,0.58,61.20,60.35,61.5,60.25,1.00,1.75,0.59,NaN,1.09
4,Indian Railway Catering & Tourism Corp.Ltd.,320,14-10-2019,637.97,127.42,626.00,727.75,743.8,625.00,105.55,351.03,13.28,0.88,109.21


In [26]:
rename_dict = {
    "company_name": "company",
    "ipo_issue_price_final": "issue_price",
    "listing_date": "date",
    "ipo_issue_size_crore": "size",
    "change_in_percentage_listing_day": "change",
    "ildt_open_price": "open",
    "ildt_close_price": "close",
    "ildt_high_price": "high",
    "ildt_low_price": "low",
    "qib": "qib",
    "nii": "nii",
    "rii": "rii",
    "emp": "emp",
    "total": "total"
}

In [27]:
ipo.rename(columns=rename_dict,inplace=True)

In [28]:
ipo.head()

,company,issue_price,date,size,change,open,close,high,low,qib,nii,rii,emp,total
0,Prince Pipes & Fittings Ltd.,178,30-12-2019,500.00,-6.24,160.00,166.90,177.9,152.50,2.02,0.52,1.63,NaN,1.66
1,Ujjivan Small Finance Bank Ltd.,37,12-12-2019,745.95,51.22,58.75,55.95,62.8,53.05,46.30,473.33,45.29,NaN,99.61
2,CSB Bank Ltd.,195,04-12-2019,409.68,53.87,275.00,300.05,306.8,275.00,25.52,164.27,40.98,NaN,47.88
3,Vishwaraj Sugar Industries Ltd.,60,15-10-2019,60.00,0.58,61.20,60.35,61.5,60.25,1.00,1.75,0.59,NaN,1.09
4,Indian Railway Catering & Tourism Corp.Ltd.,320,14-10-2019,637.97,127.42,626.00,727.75,743.8,625.00,105.55,351.03,13.28,0.88,109.21


In [29]:
ipo.isnull().sum().reset_index(name='null_values')

,index,null_values
0,company,0
1,issue_price,0
2,date,0
3,size,0
4,change,0
5,open,0
6,close,0
7,high,0
8,low,0
9,qib,0


In [30]:
calculated_change = ((ipo['close']-ipo['issue_price'])/ipo['issue_price']).mul(100).round(2)

In [31]:
changes = pd.DataFrame({
    'change_diff': ipo['change'] - calculated_change
})

changes['change_diff'].value_counts().reset_index(name = 'cnt').query('cnt>0')

,change_diff,cnt
0,0.0,302


In [32]:
ipo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   company      302 non-null    object 
 1   issue_price  302 non-null    int64  
 2   date         302 non-null    object 
 3   size         302 non-null    float64
 4   change       302 non-null    float64
 5   open         302 non-null    float64
 6   close        302 non-null    float64
 7   high         302 non-null    float64
 8   low          302 non-null    float64
 9   qib          302 non-null    float64
 10  nii          302 non-null    float64
 11  rii          302 non-null    float64
 12  emp          123 non-null    float64
 13  total        302 non-null    float64
dtypes: float64(11), int64(1), object(2)
memory usage: 33.2+ KB


In [33]:
ipo['date'] = pd.to_datetime(ipo['date'], format='%d-%m-%Y')

In [34]:
ipo.head()

,company,issue_price,date,size,change,open,close,high,low,qib,nii,rii,emp,total
0,Prince Pipes & Fittings Ltd.,178,2019-12-30,500.00,-6.24,160.00,166.90,177.9,152.50,2.02,0.52,1.63,NaN,1.66
1,Ujjivan Small Finance Bank Ltd.,37,2019-12-12,745.95,51.22,58.75,55.95,62.8,53.05,46.30,473.33,45.29,NaN,99.61
2,CSB Bank Ltd.,195,2019-12-04,409.68,53.87,275.00,300.05,306.8,275.00,25.52,164.27,40.98,NaN,47.88
3,Vishwaraj Sugar Industries Ltd.,60,2019-10-15,60.00,0.58,61.20,60.35,61.5,60.25,1.00,1.75,0.59,NaN,1.09
4,Indian Railway Catering & Tourism Corp.Ltd.,320,2019-10-14,637.97,127.42,626.00,727.75,743.8,625.00,105.55,351.03,13.28,0.88,109.21


In [35]:
ipo[ipo[['company','date']].duplicated()]

,company,issue_price,date,size,change,open,close,high,low,qib,nii,rii,emp,total


In [36]:
ipo.to_csv(os.path.join(data_pref,"clean",'ipo.csv'),index=False)

### VIX

In [37]:
vix.head()

,Price,Close,High,Low,Open
0,01-01-2019,15.330000,16.400000,15.15,16.000000
1,02-01-2019,16.389999,16.950001,14.84,15.330000
2,03-01-2019,16.790001,17.150000,15.44,16.389999
3,04-01-2019,16.160000,16.969999,15.49,16.790001
4,07-01-2019,16.299999,16.450001,15.30,16.160000


In [38]:
vix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1840 entries, 0 to 1839
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Price   1840 non-null   object 
 1   Close   1840 non-null   float64
 2   High    1840 non-null   float64
 3   Low     1840 non-null   float64
 4   Open    1840 non-null   float64
dtypes: float64(4), object(1)
memory usage: 72.0+ KB


In [39]:
vix['Price']=pd.to_datetime(vix['Price'],format="%d-%m-%Y")

In [40]:
rename_dict = {
    'Price':'date',
    'Close':'close',
    'High':'high',
    'Low':'low',
    'Open':'open'
}

vix.rename(columns=rename_dict,inplace=True)

In [41]:
vix[['close','high','low','open']] = vix[['close','high','low','open']].round(2)

In [42]:
vix.head()

,date,close,high,low,open
0,2019-01-01,15.33,16.40,15.15,16.00
1,2019-01-02,16.39,16.95,14.84,15.33
2,2019-01-03,16.79,17.15,15.44,16.39
3,2019-01-04,16.16,16.97,15.49,16.79
4,2019-01-07,16.30,16.45,15.30,16.16


In [43]:
vix.isnull().sum().reset_index(name = 'null_cnt')

,index,null_cnt
0,date,0
1,close,0
2,high,0
3,low,0
4,open,0


In [44]:
vix['date'].duplicated().sum()

np.int64(0)

In [45]:
vix.to_csv(os.path.join(data_pref,"clean",'vix.csv'),index=False)

### Nifty50

In [46]:
nifty.head()

,Price,Close,High,Low,Open,Volume
0,02-01-2019,10792.50000,10895.34961,10735.04980,10868.84961,309700
1,03-01-2019,10672.25000,10814.04980,10661.25000,10796.79980,286200
2,04-01-2019,10727.34961,10741.04980,10628.65039,10699.70020,296600
3,07-01-2019,10771.79980,10835.95020,10750.15039,10804.84961,269400
4,08-01-2019,10802.15039,10818.45020,10733.25000,10786.25000,277700


In [47]:
nifty.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1856 entries, 0 to 1855
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Price   1856 non-null   object 
 1   Close   1856 non-null   float64
 2   High    1856 non-null   float64
 3   Low     1856 non-null   float64
 4   Open    1856 non-null   float64
 5   Volume  1856 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 87.1+ KB


In [48]:
nifty.isnull().sum().reset_index(name = 'null_cnt')

,index,null_cnt
0,Price,0
1,Close,0
2,High,0
3,Low,0
4,Open,0
5,Volume,0


In [49]:
nifty.rename(columns={'Price':'date'},inplace=True)
nifty.columns = nifty.columns.str.lower()

In [50]:
nifty[['close','high','low','open']] = nifty[['close','high','low','open']].round(2)

In [51]:
nifty['date'] = pd.to_datetime(nifty['date'],format="%d-%m-%Y")

In [52]:
nifty['date'].duplicated().sum()

np.int64(0)

In [53]:
nifty.head()

,date,close,high,low,open,volume
0,2019-01-02,10792.50,10895.35,10735.05,10868.85,309700
1,2019-01-03,10672.25,10814.05,10661.25,10796.80,286200
2,2019-01-04,10727.35,10741.05,10628.65,10699.70,296600
3,2019-01-07,10771.80,10835.95,10750.15,10804.85,269400
4,2019-01-08,10802.15,10818.45,10733.25,10786.25,277700


In [54]:
nifty.to_csv(os.path.join(data_pref,"clean","nifty.csv"),index=False)